In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 验证模型1：纸牌魔术（约瑟夫环）
# ==========================================
def formula_josephus(n):
    """基于公式 J(n) = 2l + 1 求解"""
    a = int(np.floor(np.log2(n)))
    l = n - 2**a
    return 2 * l + 1

def simulate_josephus(n):
    """物理过程模拟"""
    cards = list(range(1, n + 1))
    while len(cards) > 1:
        cards.append(cards.pop(0)) # 留第一张到底部
        cards.pop(0)               # 丢弃第二张
    return cards[0]

# 验证 n 从 2 到 100
n_values = range(2, 101)
sim_results = [simulate_josephus(n) for n in n_values]
form_results = [formula_josephus(n) for n in n_values]

assert sim_results == form_results, "纸牌魔术模型推导存在偏差！"
print("✅ 模型一验证通过：公式计算与物理模拟100%重合。")

# 可视化输出：展示约瑟夫环的锯齿状分布特性
plt.figure(figsize=(10, 4))
plt.plot(n_values, form_results, marker='o', linestyle='-', color='c', markersize=4)
plt.title("Josephus Problem: Winning Position vs Total Cards (Step k=2)")
plt.xlabel("Total Cards (n)")
plt.ylabel("Winning Position J(n)")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# 验证模型2：完美洗牌同余条件
# ==========================================
def find_shuffle_order(m):
    """通过物理模拟寻找复原次数"""
    deck = list(range(m))
    original = deck.copy()
    count = 0
    while True:
        half1, half2 = deck[:m//2], deck[m//2:]
        deck = [val for pair in zip(half1, half2) for val in pair]
        count += 1
        if deck == original:
            return count

def math_shuffle_order(m):
    """通过同余方程 2^n ≡ 1 mod (m-1) 求解"""
    if m == 2: return 1
    n = 1
    while (2**n) % (m - 1) != 1:
        n += 1
    return n

# 验证 m 从 4 到 52 (仅限偶数)
m_values = range(4, 53, 2)
sim_orders = [find_shuffle_order(m) for m in m_values]
math_orders = [math_shuffle_order(m) for m in m_values]

assert sim_orders == math_orders, "洗牌同余方程推导存在偏差！"
print("✅ 模型二验证通过：同余代数解与物理洗牌周期100%重合。")

# 可视化输出（你可以截图放入作业附录）
plt.figure(figsize=(10, 4))
plt.plot(m_values, math_orders, marker='o', linestyle='-', color='b')
plt.title("Number of Perfect Shuffles to Restore Deck (Out-Shuffle)")
plt.xlabel("Deck Size (m)")
plt.ylabel("Required Shuffles (n)")
plt.grid(True)
plt.show()

In [ ]:
# ==========================================
# 实验三：灵敏度分析与稳定性检验 (物理失误的鲁棒性)
# ==========================================
import random

def imperfect_shuffle(m, error_rate=0.05):
    """
    模拟不完美洗牌：按一定概率发生失误（两张牌同时落下）
    m: 偶数张牌
    error_rate: 失误概率
    """
    deck = list(range(m))
    original = deck.copy()
    
    half1, half2 = deck[:m//2], deck[m//2:]
    
    new_deck = []
    i, j = 0, 0
    turn = 0
    while i < len(half1) and j < len(half2):
        if random.random() < error_rate:
            # 发生失误，连续落下两张同半区的牌
            if turn == 0 and i + 1 < len(half1):
                new_deck.extend([half1[i], half1[i+1]])
                i += 2
                turn = 1
            elif turn == 1 and j + 1 < len(half2):
                new_deck.extend([half2[j], half2[j+1]])
                j += 2
                turn = 0
            else:
                if turn == 0:
                    new_deck.append(half1[i])
                    i += 1
                    turn = 1
                else:
                    new_deck.append(half2[j])
                    j += 1
                    turn = 0
        else:
            if turn == 0:
                new_deck.append(half1[i])
                i += 1
                turn = 1
            else:
                new_deck.append(half2[j])
                j += 1
                turn = 0
                
    new_deck.extend(half1[i:])
    new_deck.extend(half2[j:])
    
    # 计算偏差（错位牌的比例）
    differences = sum(1 for a, b in zip(new_deck, original) if a != b)
    return differences / m

# 进行蒙特卡洛模拟
error_rates = np.linspace(0, 0.5, 20)
average_divergence = []
n_trials = 1000
m = 52 # 使用标准52张牌模拟

for rate in error_rates:
    divergence = np.mean([imperfect_shuffle(m, error_rate=rate) for _ in range(n_trials)])
    average_divergence.append(divergence)

plt.figure(figsize=(10, 4))
plt.plot(error_rates, average_divergence, marker='s', linestyle='-', color='r')
plt.title("Sensitivity Analysis of Perfect Shuffle (Deck size=52)")
plt.xlabel("Error Probability (Human mistake rate)")
plt.ylabel("Proportion of Displaced Cards")
plt.grid(True)
plt.show()

print("🔍 灵敏度分析完成：可见随着物理失误概率增加，单次洗牌导致的牌序错乱度迅速上升。")
print("结论：完美洗牌对操作的精确度要求极高，系统缺乏针对人为失误的鲁棒性。")

In [ ]:
# ==========================================
# 实验四：纸牌魔术稳定性检验 (丢弃步长 k = 3)
# ==========================================
# 探索不同的留牌/丢弃步长下的行为
def extended_josephus(n, k=3):
    """
    更一般的约瑟夫环 (每次跳过 k-1 张，移除第 k 张)
    """
    cards = list(range(1, n + 1))
    pos = 0
    while len(cards) > 1:
        # 下一个要移除的位置
        pos = (pos + k - 1) % len(cards)
        cards.pop(pos)
    return cards[0]

n_values_ext = range(1, 150)
k3_results = [extended_josephus(n, k=3) for n in n_values_ext]

plt.figure(figsize=(10, 4))
plt.scatter(n_values_ext, k3_results, color='green', marker='.')
plt.plot(n_values_ext, k3_results, color='green', alpha=0.3)
plt.title("Josephus Position for Step k=3")
plt.xlabel("Total Cards (n)")
plt.ylabel("Winning Position")
plt.grid(True)
plt.show()

print("🔍 稳定性检验：可以看到当 k > 2 时，结果呈现出不规则的分形锯齿结构，虽然仍然具有递推性，但在 n 分解上变得更加复杂。")